In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from google.colab import drive

In [ ]:
file_path = "/content/drive/MyDrive/Analisis matemático/FINAL 2027 CSV.csv"
df_final = pd.read_csv(file_path, encoding='latin1')



In [ ]:
df_final

,COUNTRY,YEAR,POBLACION_0_14,POBLACION_TOTAL,PIB_POR_PAIS,TASA_MORTALIDAD_INFANTIL,POBLACION_20_24_VARONES,POBLACION_20_24_MUJERES,MEDALLAS_TOTALES_POR_PAIS_POR_AÑO,MEDALLAS_DE_ORO_TOTALES_POR_PAIS_POR_AÑO,...,POBLACION_20_24_CENTRADO,PIB_POR_PAIS_CENTRADO,TASA_MORTALIDAD_INFANTIL_CENTRADO,LOCAL,VECINO,INDICE_PRECIOS,PIB_REAL,LOG_PIB_REAL,PROP_MEDALLAS,LOG_PROP_MEDALLAS
0,Afghanistan,1960,3789907,9035043.0,NaN,NaN,8.97,8.80,0,0,...,-8.023285e+05,NaN,NaN,0,0,0.10,NaN,NaN,0.00000,NaN
1,Afghanistan,1964,4162207,9814318.0,NaN,223.9,8.84,8.71,0,0,...,-6.974664e+05,NaN,131.826154,0,0,0.13,NaN,NaN,0.00000,NaN
2,Afghanistan,1968,4671648,10756922.0,NaN,209.3,8.79,9.07,0,0,...,-7.419265e+05,NaN,123.978873,0,0,0.16,NaN,NaN,0.00000,NaN
3,Afghanistan,1972,5261617,11853696.0,NaN,194.8,8.34,8.68,0,0,...,-1.150589e+06,NaN,116.710828,0,0,0.20,NaN,NaN,0.00000,NaN
4,Afghanistan,1976,5909060,13059851.0,NaN,180.0,8.31,8.40,0,0,...,-1.420432e+06,NaN,109.580000,0,0,0.25,NaN,NaN,0.00000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3480,Zimbabwe,2008,5489264,12959149.0,340.740183,55.6,11.11,11.66,4,1,...,-2.869097e+06,-16603.48877,26.806218,0,0,0.90,378.600203,5.936481,0.00196,-6.234901
3481,Zimbabwe,2012,6035612,13817887.0,1238.601090,46.8,9.05,9.34,0,0,...,-3.489705e+06,-15982.70382,21.544041,0,0,0.95,1303.790621,7.173031,0.00000,NaN
3482,Zimbabwe,2016,6365033,14600294.0,1407.415364,40.4,7.89,8.19,0,0,...,-3.510867e+06,-15073.80959,17.708808,0,0,0.98,1436.138127,7.269713,0.00000,NaN
3483,Zimbabwe,2020,6549831,15526888.0,1730.413489,36.4,8.56,8.52,0,0,...,-3.221034e+06,-15386.28402,15.907254,0,0,1.00,1730.413489,7.456116,0.00000,NaN


In [ ]:
import numpy as np

# 1. Calcular total de medallas de oro entregadas por año
oro_total_por_año = df_final.groupby("YEAR")["MEDALLAS_DE_ORO_TOTALES_POR_PAIS_POR_AÑO"].transform("sum")

# 2. Calcular proporción de medallas de oro por país
df_final["PROP_MEDALLAS_ORO"] = df_final["MEDALLAS_DE_ORO_TOTALES_POR_PAIS_POR_AÑO"] / oro_total_por_año

# 3. Reemplazar ceros para evitar log(0)
df_final["PROP_MEDALLAS_ORO"] = df_final["PROP_MEDALLAS_ORO"].replace(0, 1e-6)

# 4. Calcular logaritmo de la proporción
df_final["LOG_PROP_MEDALLAS_ORO"] = np.log(df_final["PROP_MEDALLAS_ORO"])


In [ ]:
# Verificar que la suma de proporciones por año sea 1 (o muy cercano)
suma_por_anio = df_final.groupby("YEAR")["PROP_MEDALLAS_ORO"].sum()
print(suma_por_anio.head(10))  # O toda si prefieres

# ¿Hay años muy diferentes de 1?
print(suma_por_anio.describe())

# Mayores proporciones
print(df_final[["COUNTRY", "YEAR", "PROP_MEDALLAS_ORO", "LOG_PROP_MEDALLAS_ORO"]]
      .sort_values("PROP_MEDALLAS_ORO", ascending=False).head(10))

# Menores proporciones (distintos de 0)
print(df_final[df_final["PROP_MEDALLAS_ORO"] > 0]
      [["COUNTRY", "YEAR", "PROP_MEDALLAS_ORO", "LOG_PROP_MEDALLAS_ORO"]]
      .sort_values("PROP_MEDALLAS_ORO").head(10))



YEAR
1960    1.000182
1964    1.000180
1968    1.000175
1972    1.000181
1976    1.000180
1980    1.000180
1984    1.000180
1988    1.000175
1992    1.000168
1996    1.000153
Name: PROP_MEDALLAS_ORO, dtype: float64
count    17.000000
mean      1.000164
std       0.000015
min       1.000143
25%       1.000150
50%       1.000168
75%       1.000180
max       1.000182
Name: PROP_MEDALLAS_ORO, dtype: float64
            COUNTRY  YEAR  PROP_MEDALLAS_ORO  LOG_PROP_MEDALLAS_ORO
2606         Russia  1980           0.409190              -0.893575
3338  United States  1984           0.374245              -0.982843
1160        Germany  1976           0.280822              -1.270035
3334  United States  1968           0.275766              -1.288203
3333  United States  1964           0.273775              -1.295448
2604         Russia  1972           0.264851              -1.328586
3332  United States  1960           0.262136              -1.338892
3341  United States  1996           0.261513     

In [ ]:
# Crear una función para buscar el valor de mortalidad de 2020 o 2016
def imputar_mortalidad(row):
    if pd.isna(row["TASA_MORTALIDAD_INFANTIL_CENTRADO"]) and row["YEAR"] == 2024:
        # Buscar el país en 2020
        valor_2020 = df_modelo[(df_modelo["COUNTRY"] == row["COUNTRY"]) & (df_modelo["YEAR"] == 2020)]["TASA_MORTALIDAD_INFANTIL_CENTRADO"]
        if not valor_2020.empty:
            return valor_2020.values[0]
        else:
            # Buscar el país en 2016
            valor_2016 = df_modelo[(df_modelo["COUNTRY"] == row["COUNTRY"]) & (df_modelo["YEAR"] == 2016)]["TASA_MORTALIDAD_INFANTIL_CENTRADO"]
            if not valor_2016.empty:
                return valor_2016.values[0]
            else:
                # Si no hay 2020 ni 2016, imputar 0
                return 0
    else:
        return row["TASA_MORTALIDAD_INFANTIL_CENTRADO"]

# Aplicar la imputación
df_final["TASA_MORTALIDAD_INFANTIL_CENTRADO"] = df_final.apply(imputar_mortalidad, axis=1)


In [ ]:
import statsmodels.api as sm
import numpy as np

# 1. Filtrar países que ganaron al menos una medalla (para evitar log(0))
df_modelo = df_final[df_final["PROP_MEDALLAS"] > 0].copy()

# 2. Definir variable dependiente (y)
y = df_modelo["LOG_PROP_MEDALLAS_ORO"]

# 3. Definir variables explicativas (X)
X = df_modelo[[
    "LOG_PIB_REAL",
    "POBLACION_20_24_CENTRADO",
    "TASA_MORTALIDAD_INFANTIL_CENTRADO",
    "LOCAL",
    "VECINO"
]]

# 4. Agregar constante para el intercepto en la regresión
X = sm.add_constant(X)

# 5. Verificación rápida
print("Shape de X:", X.shape)
print("Shape de y:", y.shape)
X.head()


Shape de X: (1059, 6)
Shape de y: (1059,)


,const,LOG_PIB_REAL,POBLACION_20_24_CENTRADO,TASA_MORTALIDAD_INFANTIL_CENTRADO,LOCAL,VECINO
12,1.0,6.050083,-1.362870e+06,40.806218,0,0
13,1.0,6.530443,-5.452886e+05,34.844041,0,0
33,1.0,9.027067,-5.576904e+06,0.000000,0,0
40,1.0,8.883573,-2.289790e+05,10.795135,0,0
42,1.0,8.190184,3.144951e+05,-5.243523,0,0


In [ ]:
# Imputar los NaN en TASA_MORTALIDAD_INFANTIL_CENTRADO con el promedio de su año
df_modelo["TASA_MORTALIDAD_INFANTIL_CENTRADO"] = df_modelo.groupby("YEAR")["TASA_MORTALIDAD_INFANTIL_CENTRADO"].transform(lambda x: x.fillna(x.mean()))


In [ ]:
# Impute missing values for all relevant columns based on group-wise mean
for column in ["LOG_PIB_REAL", "POBLACION_20_24_CENTRADO", "TASA_MORTALIDAD_INFANTIL_CENTRADO"]:
    # Group by 'YEAR' and fill NaN with the mean of the group
    df_modelo[column] = df_modelo.groupby("YEAR")[column].transform(lambda x: x.fillna(x.mean()))



In [ ]:
# Check for NaNs in all relevant columns after imputation
for column in ["LOG_PIB_REAL", "POBLACION_20_24_CENTRADO", "TASA_MORTALIDAD_INFANTIL_CENTRADO"]:
    print(f"NaNs in {column}: {df_modelo[column].isna().sum()}")



NaNs in LOG_PIB_REAL: 0
NaNs in POBLACION_20_24_CENTRADO: 0
NaNs in TASA_MORTALIDAD_INFANTIL_CENTRADO: 0


In [ ]:
df_modelo

,COUNTRY,YEAR,POBLACION_0_14,POBLACION_TOTAL,PIB_POR_PAIS,TASA_MORTALIDAD_INFANTIL,POBLACION_20_24_VARONES,POBLACION_20_24_MUJERES,MEDALLAS_TOTALES_POR_PAIS_POR_AÑO,MEDALLAS_DE_ORO_TOTALES_POR_PAIS_POR_AÑO,...,TASA_MORTALIDAD_INFANTIL_CENTRADO,LOCAL,VECINO,INDICE_PRECIOS,PIB_REAL,LOG_PIB_REAL,PROP_MEDALLAS,LOG_PROP_MEDALLAS,PROP_MEDALLAS_ORO,LOG_PROP_MEDALLAS_ORO
12,Afghanistan,2008,13043065,26482622.0,381.733238,69.6,8.50,8.33,1,0,...,40.806218,0,0,0.90,424.148042,6.050083,0.000490,-7.621195,0.000001,-13.815511
13,Afghanistan,2012,14548198,30560034.0,651.417134,60.1,9.06,8.89,1,0,...,34.844041,0,0,0.95,685.702247,6.530443,0.000517,-7.567346,0.000001,-13.815511
33,Albania,2024,467968,2745972.0,8575.171134,NaN,7.40,6.80,2,0,...,0.000000,0,0,1.03,8325.408868,9.027067,0.000883,-7.032183,0.000001,-13.815511
40,Algeria,1984,9580584,21271969.0,2524.380714,67.7,9.51,9.02,2,0,...,10.795135,0,0,0.35,7212.516326,8.883573,0.001356,-6.603266,0.000001,-13.815511
42,Algeria,1992,10910578,26628568.0,1802.693008,42.1,9.95,9.53,2,1,...,-5.243523,0,0,0.50,3605.386016,8.190184,0.001184,-6.738745,0.001789,-6.326149
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3460,Zambia,1996,4343670,9004053.0,399.511305,100.3,9.72,9.46,1,0,...,56.864249,0,0,0.60,665.852175,6.501068,0.000543,-7.518064,0.000001,-13.815511
3467,Zambia,2024,8694765,20723965.0,1330.727806,NaN,9.41,9.37,1,0,...,0.000000,0,0,1.03,1291.968744,7.163922,0.000442,-7.725330,0.000001,-13.815511
3473,Zimbabwe,1980,3428620,7041303.0,948.527311,70.5,7.87,9.40,15,15,...,6.265143,0,0,0.30,3161.757703,8.058883,0.010830,-4.525405,0.032823,-3.416633
3479,Zimbabwe,2004,5154172,12365896.0,469.484654,51.6,11.68,12.62,3,1,...,18.489637,0,0,0.80,586.855817,6.374779,0.001506,-6.498282,0.001511,-6.495266


In [ ]:
# 1. Verificar que ya no haya NaN en TASA_MORTALIDAD_INFANTIL_CENTRADO
nulos = df_modelo["TASA_MORTALIDAD_INFANTIL_CENTRADO"].isna().sum()
print(f"Número de valores NaN en TASA_MORTALIDAD_INFANTIL_CENTRADO: {nulos}")

# 2. Verificar algunos ejemplos de países en 2024
ejemplos_2024 = df_modelo[df_modelo["YEAR"] == 2024][[
    "COUNTRY", "YEAR", "TASA_MORTALIDAD_INFANTIL_CENTRADO"
]].sample(10, random_state=42)
print("Ejemplos de mortalidad infantil imputada en 2024:")
print(ejemplos_2024)

# 3. Resumen estadístico rápido
print("\nResumen estadístico de TASA_MORTALIDAD_INFANTIL_CENTRADO:")
print(df_modelo["TASA_MORTALIDAD_INFANTIL_CENTRADO"].describe())


Número de valores NaN en TASA_MORTALIDAD_INFANTIL_CENTRADO: 0
Ejemplos de mortalidad infantil imputada en 2024:
          COUNTRY  YEAR  TASA_MORTALIDAD_INFANTIL_CENTRADO
3059  Switzerland  2024                         -16.992746
33        Albania  2024                           0.000000
2872     Slovenia  2024                         -18.592746
849       Denmark  2024                         -17.292746
475      Bulgaria  2024                         -15.092746
2277  New Zealand  2024                         -16.492746
424      Botswana  2024                          12.207254
764       Croatia  2024                         -16.492746
152         Aruba  2024                           0.000000
2600      Romania  2024                         -15.092746

Resumen estadístico de TASA_MORTALIDAD_INFANTIL_CENTRADO:
count    1059.000000
mean      -18.066310
std        28.641163
min       -78.900855
25%       -35.283834
50%       -19.193782
75%        -7.242746
max       101.826154
Name: TASA_M

In [ ]:
# Definir X e y ahora que todo está limpio
X = df_modelo[[
    "LOG_PIB_REAL",
    "POBLACION_20_24_CENTRADO",
    "TASA_MORTALIDAD_INFANTIL_CENTRADO",
    "LOCAL",
    "VECINO"
]]

# Agregar constante
X = sm.add_constant(X)

# Definir y
y = df_modelo["LOG_PROP_MEDALLAS_ORO"]

# Confirmar shapes
print("Shape final de X:", X.shape)
print("Shape final de y:", y.shape)
X.head()


Shape final de X: (1059, 6)
Shape final de y: (1059,)


,const,LOG_PIB_REAL,POBLACION_20_24_CENTRADO,TASA_MORTALIDAD_INFANTIL_CENTRADO,LOCAL,VECINO
12,1.0,6.050083,-1.362870e+06,40.806218,0,0
13,1.0,6.530443,-5.452886e+05,34.844041,0,0
33,1.0,9.027067,-5.576904e+06,0.000000,0,0
40,1.0,8.883573,-2.289790e+05,10.795135,0,0
42,1.0,8.190184,3.144951e+05,-5.243523,0,0


In [ ]:
# Ajustar el modelo log-lineal
modelo = sm.OLS(y, X).fit()

# Mostrar resumen del modelo
print(modelo.summary())


                              OLS Regression Results                             
Dep. Variable:     LOG_PROP_MEDALLAS_ORO   R-squared:                       0.122
Model:                               OLS   Adj. R-squared:                  0.117
Method:                    Least Squares   F-statistic:                     29.15
Date:                   Wed, 30 Apr 2025   Prob (F-statistic):           9.01e-28
Time:                           01:04:00   Log-Likelihood:                -3016.0
No. Observations:                   1059   AIC:                             6044.
Df Residuals:                       1053   BIC:                             6074.
Df Model:                              5                                         
Covariance Type:               nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------

In [ ]:
import numpy as np

# 1. Predecir log(ŵ_it) con el modelo
log_predicciones = modelo.predict(X)

# 2. Destransformar: volver a escala real (ŵ_it)
predicciones = np.exp(log_predicciones)

# 3. Agregar las predicciones al dataframe de modelado
df_modelo["PROP_MEDALLAS_ORO_PREDICHA"] = predicciones

# 4. Opcional: calcular error absoluto entre proporción real y predicha
df_modelo["ERROR_ABSOLUTO"] = np.abs(df_modelo["PROP_MEDALLAS_ORO"] - df_modelo["PROP_MEDALLAS_ORO_PREDICHA"])

# 5. Mostrar ejemplo de tabla de resultados
resultados = df_modelo[[
    "COUNTRY",
    "YEAR",
    "MEDALLAS_DE_ORO_TOTALES_POR_PAIS_POR_AÑO",
    "PROP_MEDALLAS_ORO",
    "PROP_MEDALLAS_ORO_PREDICHA",
    "ERROR_ABSOLUTO"
]].sort_values(by="ERROR_ABSOLUTO", ascending=False)

# Mostrar los 10 países con mayor error
resultados.head(10)


,COUNTRY,YEAR,MEDALLAS_DE_ORO_TOTALES_POR_PAIS_POR_AÑO,PROP_MEDALLAS_ORO,PROP_MEDALLAS_ORO_PREDICHA,ERROR_ABSOLUTO
658,China,2008,74,0.110283,0.477340,0.367056
2606,Russia,1980,187,0.409190,0.042577,0.366614
1160,Germany,1976,123,0.280822,0.001726,0.279096
3333,United States,1964,95,0.273775,0.003588,0.270188
3334,United States,1968,99,0.275766,0.010362,0.265404
2604,Russia,1972,107,0.264851,0.001007,0.263845
2605,Russia,1976,114,0.260274,0.001337,0.258937
3332,United States,1960,81,0.262136,0.003614,0.258521
2608,Russia,1988,134,0.257692,0.000647,0.257045
1161,Germany,1980,115,0.251641,0.002060,0.249581


In [ ]:
# 1. Primero, recuperar el total de medallas entregadas cada año
total_medallas_por_ano = df_modelo.groupby("YEAR")["MEDALLAS_TOTALES_POR_PAIS_POR_AÑO"].transform("sum")

# 2. Calcular las medallas reales y predichas en números absolutos
df_modelo["MEDALLAS_REALES"] = df_modelo["PROP_MEDALLAS"] * total_medallas_por_ano
df_modelo["MEDALLAS_PREDICHAS"] = df_modelo["PROP_MEDALLAS_PREDICHA"] * total_medallas_por_ano

# 3. Calcular error absoluto en número de medallas
df_modelo["ERROR_ABSOLUTO_MEDALLAS"] = np.abs(df_modelo["MEDALLAS_REALES"] - df_modelo["MEDALLAS_PREDICHAS"])

# 4. Armar la tabla ordenada
tabla_resultados = df_modelo[[
    "COUNTRY",
    "YEAR",
    "MEDALLAS_REALES",
    "MEDALLAS_PREDICHAS",
    "ERROR_ABSOLUTO_MEDALLAS"
]].sort_values(by="ERROR_ABSOLUTO_MEDALLAS", ascending=False)

# 5. Mostrar los 10 países con mayor error absoluto
tabla_resultados.head(10)


,COUNTRY,YEAR,MEDALLAS_REALES,MEDALLAS_PREDICHAS,ERROR_ABSOLUTO_MEDALLAS
658,China,2008,183.999999,744.192233,560.192234
2606,Russia,1980,442.000000,87.883185,354.116815
3348,United States,2024,320.999999,23.730637,297.269363
3344,United States,2008,317.000001,21.708191,295.291810
2608,Russia,1988,300.000001,10.427995,289.572005
1163,Germany,1988,295.999999,15.782880,280.217119
3347,United States,2020,298.000001,22.750077,275.249924
2605,Russia,1976,286.000000,12.553282,273.446719
1160,Germany,1976,273.000000,13.375014,259.624987
1161,Germany,1980,263.999999,15.044010,248.955990


In [ ]:
tabla_resultados

,COUNTRY,YEAR,MEDALLAS_REALES,MEDALLAS_PREDICHAS,ERROR_ABSOLUTO_MEDALLAS
658,China,2008,183.999999,744.194521,560.194522
2606,Russia,1980,442.000000,87.882760,354.117240
3348,United States,2024,320.999999,23.730753,297.269247
3344,United States,2008,317.000001,21.708214,295.291787
2608,Russia,1988,300.000001,10.427910,289.572090
...,...,...,...,...,...
2298,Niger,1972,1.000000,0.947165,0.052834
3058,Switzerland,2020,15.000001,15.039850,0.039849
2142,Morocco,1960,1.000000,0.966093,0.033907
1454,Indonesia,1996,5.999999,6.015079,0.015080


In [ ]:
tabla_resultados.to_excel('RESULTADOS.xlsx', index=False)

In [ ]:
import statsmodels.api as sm

# 1. Crear Dummies para las grandes potencias
df_modelo["COUNTRY"] = df_modelo["COUNTRY"].str.strip()

df_modelo["DUMMY_CHINA"] = df_modelo["COUNTRY"].apply(lambda x: 1 if x == "China" else 0)
df_modelo["DUMMY_RUSSIA"] = df_modelo["COUNTRY"].apply(lambda x: 1 if x in ["Russia", "Soviet Union", "USSR"] else 0)
df_modelo["DUMMY_USA"] = df_modelo["COUNTRY"].apply(lambda x: 1 if x == "United States" else 0)
df_modelo["DUMMY_GERMANY"] = df_modelo["COUNTRY"].apply(lambda x: 1 if x == "Germany" else 0)

# 2. Redefinir X incluyendo las nuevas variables
X_potencias = df_modelo[[
    "LOG_PIB_REAL",
    "POBLACION_20_24_CENTRADO",
    "TASA_MORTALIDAD_INFANTIL_CENTRADO",
    "LOCAL",
    "VECINO",
    "DUMMY_CHINA",
    "DUMMY_RUSSIA",
    "DUMMY_USA",
    "DUMMY_GERMANY"
]]

# 3. Agregar constante
X_potencias = sm.add_constant(X_potencias)

# 4. Definir y (como antes)
y_potencias = df_modelo["LOG_PROP_MEDALLAS_ORO"]

# 5. Ajustar el nuevo modelo
modelo_potencias = sm.OLS(y_potencias, X_potencias).fit()

# 6. Mostrar resumen del modelo ajustado
print(modelo_potencias.summary())


                              OLS Regression Results                             
Dep. Variable:     LOG_PROP_MEDALLAS_ORO   R-squared:                       0.173
Model:                               OLS   Adj. R-squared:                  0.166
Method:                    Least Squares   F-statistic:                     24.44
Date:                   Wed, 30 Apr 2025   Prob (F-statistic):           2.82e-38
Time:                           01:10:13   Log-Likelihood:                -2983.9
No. Observations:                   1059   AIC:                             5988.
Df Residuals:                       1049   BIC:                             6037.
Df Model:                              9                                         
Covariance Type:               nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------